In [1]:
# ===============================
# 1. Import libraries
# ===============================

import tensorflow as tf
import numpy as np
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

I0000 00:00:1780590332.402022   16047 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780590332.871378   16047 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780590334.767638   16047 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:

# ===============================
# 2. Create a small text dataset
# ===============================

texts = [
    "The movie was fantastic and I loved it",
    "This film was amazing and very enjoyable",
    "I really liked the story and acting",
    "The movie was beautiful and emotional",
    "Excellent film with great performance",
    "I enjoyed every moment of the movie",
    "The acting was wonderful",
    "The story was interesting and powerful",
    "It was a great experience",
    "I would watch this movie again",

    "The movie was terrible and boring",
    "I hated this film",
    "The story was weak and disappointing",
    "The acting was very bad",
    "This was a waste of time",
    "I did not enjoy the movie",
    "The film was dull and slow",
    "Very poor performance",
    "The movie was not good",
    "I will never watch this again"
]

# 1 = positive, 0 = negative
labels = [
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0
]

texts = np.array(texts)
labels = np.array(labels)

In [7]:
# ===============================
# 3. Train-test split
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)
# Important fix for string dtype issue
X_train = tf.constant(X_train, dtype=tf.string)
X_test = tf.constant(X_test, dtype=tf.string)

y_train = tf.constant(y_train, dtype=tf.float32)
y_test = tf.constant(y_test, dtype=tf.float32)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 16
Testing samples: 4


In [8]:
# ===============================
# 4. TextVectorization layer
# ===============================

max_tokens = 1000       # maximum vocabulary size
sequence_length = 10    # every sentence will become 10 words/tokens long

vectorizer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=sequence_length
)

# Learn vocabulary from training text
vectorizer.adapt(X_train)

# Check vocabulary
print(vectorizer.get_vocabulary()[:20])

['', '[UNK]', np.str_('was'), np.str_('the'), np.str_('and'), np.str_('movie'), np.str_('i'), np.str_('this'), np.str_('story'), np.str_('film'), np.str_('acting'), np.str_('watch'), np.str_('of'), np.str_('great'), np.str_('again'), np.str_('a'), np.str_('would'), np.str_('wonderful'), np.str_('with'), np.str_('will')]


In [9]:
# ===============================
# 5. Build the model
# ===============================

model = tf.keras.Sequential([
    vectorizer,

    layers.Embedding(
        input_dim=max_tokens,
        output_dim=16
    ),

    layers.GlobalAveragePooling1D(),

    layers.Dense(16, activation="relu"),

    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ ?                      │   0 (unbuilt) │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
# ===============================
# 6. Train the model
# ===============================

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    validation_data=(X_test, y_test)
)

Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4375 - loss: 0.6929 - val_accuracy: 0.2500 - val_loss: 0.6967
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.5000 - loss: 0.6924 - val_accuracy: 0.2500 - val_loss: 0.6968
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.6250 - loss: 0.6920 - val_accuracy: 0.2500 - val_loss: 0.6970
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.6250 - loss: 0.6916 - val_accuracy: 0.2500 - val_loss: 0.6973
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.6250 - loss: 0.6912 - val_accuracy: 0.2500 - val_loss: 0.6976
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6250 - loss: 0.6908 - val_accuracy: 0.2500 - val_loss: 0.6977
Epoch 7/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.6875 - loss: 0.6903 - val_accuracy: 0.0000e+00 - val_loss: 0.6979
Epoch 8/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.6875 - loss: 0.6899 - val_accuracy: 0.2500 - val_loss: 0.698

In [11]:
# ===============================
# 7. Evaluate the model
# ===============================

loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.2500 - loss: 0.7031
Test Loss: 0.7030643224716187
Test Accuracy: 0.25


In [13]:
# ===============================
# 8. Predict new sentences
# ===============================

def predict_sentiment(sentence):
    sentence = tf.constant([sentence], dtype=tf.string)

    prediction = model.predict(sentence)[0][0]

    if prediction >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    print("Prediction score:", prediction)
    print("Sentiment:", sentiment)

predict_sentiment("The movie was really good and enjoyable")
predict_sentiment("The film was boring and bad")
predict_sentiment("I loved the acting")
predict_sentiment("I did not like the story")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
Prediction score: 0.49514753
Sentiment: Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Prediction score: 0.48682314
Sentiment: Negative
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Prediction score: 0.50745803
Sentiment: Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Prediction score: 0.5035855
Sentiment: Positive
